# MetroPT-3 dataset exploration

This notebook loads the original MetroPT-3 CSV, checks its structure, and plots the sensor signals. It does not build a model.

Dataset: [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/791/metropt%2B3%2Bdataset) (CC BY 4.0).

## 1. Get the data

The original CSV is kept in `data/raw/`. This cell downloads it only when it is not already there. The CSV is not edited.

In [ ]:
from pathlib import Path
import zipfile
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RAW_DIR = ROOT / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = RAW_DIR / 'MetroPT3(AirCompressor).csv'
ZIP_PATH = RAW_DIR / 'metropt3_uci.zip'
UCI_ZIP_URL = 'https://archive.ics.uci.edu/static/public/791/metropt+3+dataset.zip'

if CSV_PATH.exists():
    print(f'Using existing raw dataset: {CSV_PATH}')
elif ZIP_PATH.exists():
    print(f'Using existing UCI archive: {ZIP_PATH}')
else:
    print('Downloading the original UCI archive...')
    urlretrieve(UCI_ZIP_URL, ZIP_PATH)

if not CSV_PATH.exists():
    with zipfile.ZipFile(ZIP_PATH) as archive:
        csv_member = next(name for name in archive.namelist() if name.lower().endswith('.csv'))
        with archive.open(csv_member) as source, CSV_PATH.open('wb') as target:
            target.write(source.read())

df = pd.read_csv(CSV_PATH)
print(f'Loaded {CSV_PATH.name}: {df.shape[0]:,} rows × {df.shape[1]} columns')

## 2. Explore the data

In [ ]:
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
display(df.dtypes.rename('data type').to_frame())
display(df.head())
display(df.tail())
display(df.isna().sum().rename('missing values').to_frame())
display(df.describe(include='all').T)
display(df.nunique(dropna=False).rename('unique values').to_frame())

## 3. Time column and sampling interval

The timestamp is converted to datetime and the rows are sorted before plotting. The median interval gives a quick estimate of the sampling step.

In [ ]:
timestamp_col = next((col for col in df.columns if col.lower() == 'timestamp'), None)
if timestamp_col is None:
    raise KeyError('No timestamp column was found.')

df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors='coerce')
df = df.sort_values(timestamp_col).reset_index(drop=True)
time_diffs = df[timestamp_col].diff().dropna()
print('Timestamp column:', timestamp_col)
print('First timestamp:', df[timestamp_col].min())
print('Last timestamp:', df[timestamp_col].max())
print('Median sampling interval:', time_diffs.median())
print('Most common intervals:')
display(time_diffs.value_counts().head(10).rename_axis('interval').to_frame('count'))
print('Rows with missing timestamps:', df[timestamp_col].isna().sum())

## 3.1 Column definitions

These descriptions follow the [UCI dataset documentation](https://archive.ics.uci.edu/dataset/791/metropt%2B3%2Bdataset). The downloaded CSV includes a row identifier and timestamp plus 15 sensor channels.

| Column | Definition |
|---|---|
| `Unnamed: 0` | Row identifier supplied with the dataset. |
| `timestamp` | Date and time when the sensor readings were recorded. |
| `TP2` | Pressure measured at the compressor (bar). |
| `TP3` | Pressure generated at the pneumatic panel (bar). |
| `H1` | Pressure associated with the pressure drop when the cyclonic separator filter discharges (bar). |
| `DV_pressure` | Pressure drop when the air-dryer towers discharge; zero indicates the compressor is operating under load (bar). |
| `Reservoirs` | Downstream reservoir pressure, expected to be close to `TP3` (bar). |
| `Oil_temperature` | Compressor oil temperature (°C). |
| `Motor_current` | Current in one phase of the three-phase motor (A). UCI describes values near 0 A as off, 4 A as offloaded, 7 A as under load, and 9 A during startup. |
| `COMP` | Compressor air-intake valve electrical signal. Active means no air intake, so the compressor is off or offloaded. |
| `DV_eletric` | Compressor outlet valve electrical signal. Active means the compressor is under load; inactive means it is off or offloaded. |
| `Towers` | Selects the air-drying tower: inactive means tower 1, active means tower 2. |
| `MPG` | Signal that starts the compressor under load when APU pressure falls below 8.2 bar; UCI says it follows the `COMP` sensor behavior. |
| `LPS` | Signal that activates when pressure drops below 7 bar. |
| `Pressure_switch` | Electrical signal indicating discharge in the air-drying towers. |
| `Oil_level` | Electrical signal that activates when compressor oil is below the expected level. |
| `Caudal_impulses` | Pulse count representing air flow from the APU to the reservoirs. |

The exact column names above match the downloaded CSV.

## 4. Signal summary

The continuous signals are the numeric measurement channels. State signals have at most two observed values. The timestamp and row identifier are metadata.

In [ ]:
index_cols = [col for col in df.columns if col.lower() in {'index', 'unnamed: 0'}]
numeric_cols = df.select_dtypes(include='number').columns.tolist()
signal_cols = [col for col in numeric_cols if col not in index_cols]
state_cols = [col for col in signal_cols if df[col].nunique(dropna=True) <= 2]
continuous_cols = [col for col in signal_cols if col not in state_cols]

summary = pd.DataFrame(index=df.columns)
summary['data type'] = df.dtypes.astype(str)
summary['missing values'] = df.isna().sum()
summary['min'] = df.min(numeric_only=True).reindex(df.columns)
summary['max'] = df.max(numeric_only=True).reindex(df.columns)
summary['mean'] = df.mean(numeric_only=True).reindex(df.columns)
summary['median'] = df.median(numeric_only=True).reindex(df.columns)
summary['standard deviation'] = df.std(numeric_only=True).reindex(df.columns)
summary['unique values'] = df.nunique(dropna=False)
display(summary)
print('Timestamp:', timestamp_col)
print('Continuous sensor signals:', continuous_cols)
print('Binary/state signals:', state_cols)
print('Index columns:', index_cols)

### What I see

UCI describes analogue measurements such as pressure, motor current, and oil temperature, alongside digital state signals. The summary above shows which columns vary continuously and which have only a few states in this file.

## 5. Plot sensor signals over time

The plots use 5-minute means to make the long recording quicker to inspect. The raw dataframe stays unchanged. Set `signals` to a short list (for example `['TP2', 'TP3', 'H1']`) to focus on selected channels. Leave it empty to plot all continuous signals. The Plotly toolbar supports zoom, pan, reset, and hover values. All rows share the same time axis.

In [ ]:
signals = []  # Example: ['TP2', 'TP3', 'H1']
plot_df = (
    df.set_index(timestamp_col)[continuous_cols]
      .resample('5min')
      .mean()
      .dropna(how='all')
      .reset_index()
)
selected_signals = signals or continuous_cols
unknown_signals = sorted(set(selected_signals) - set(continuous_cols))
if unknown_signals:
    raise ValueError(f'Not continuous signals: {unknown_signals}')
if not selected_signals:
    raise ValueError('No continuous signals were found to plot.')

fig = make_subplots(
    rows=len(selected_signals), cols=1, shared_xaxes=True,
    vertical_spacing=min(0.04, 0.5 / max(len(selected_signals), 1)),
    subplot_titles=selected_signals
)
for row, signal in enumerate(selected_signals, start=1):
    fig.add_trace(
        go.Scattergl(x=plot_df[timestamp_col], y=plot_df[signal], mode='lines', name=signal,
                     line=dict(width=1), showlegend=False),
        row=row, col=1
    )
fig.update_yaxes(title_text='5-minute mean')
fig.update_xaxes(title_text='Time', showspikes=True)
fig.update_layout(
    height=max(450, 220 * len(selected_signals)),
    title='MetroPT-3 continuous signals (5-minute means)', hovermode='x',
    template='plotly_white', margin=dict(l=60, r=25, t=70, b=45),
    legend=dict(orientation='h')
)
fig.show()

## 6. Correlation between continuous signals

Pearson correlation compares pairs of continuous signals. A high correlation shows that two signals move together in this dataset; it does not mean one causes the other.

In [ ]:
corr = df[continuous_cols].corr(method='pearson')
fig_corr = px.imshow(
    corr, text_auto='.2f', aspect='auto', color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1, origin='upper', labels=dict(color='Pearson r'),
    title='Pearson correlation between continuous signals'
)
fig_corr.update_layout(template='plotly_white', height=max(500, 45 * len(continuous_cols)))
fig_corr.show()

# Show distinct signal pairs only.
pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
pair_table = pairs.rename('Pearson r').reset_index()
pair_table.columns = ['signal 1', 'signal 2', 'Pearson r']
strong_positive = pair_table[pair_table['Pearson r'] >= 0.7].sort_values('Pearson r', ascending=False)
strong_negative = pair_table[pair_table['Pearson r'] <= -0.7].sort_values('Pearson r')
weak = pair_table[pair_table['Pearson r'].abs() < 0.2].sort_values('Pearson r', key=lambda x: x.abs())
print('Strong positive correlations (r ≥ 0.7):')
display(strong_positive)
print('Strong negative correlations (r ≤ -0.7):')
display(strong_negative)
print('Weak correlations (|r| < 0.2):')
display(weak)

spread = df[continuous_cols].std()
scale = df[continuous_cols].abs().mean().replace(0, np.nan)
near_constant = spread[spread.fillna(0) <= 1e-6 * scale.fillna(1)]
print('Signals with near-zero spread:')
display(near_constant.rename('standard deviation').to_frame())

## Operating Behavior Observations

I reviewed the raw summary and 5-minute means. The raw data has 1,516,948 rows, covers 1 February to 1 September 2020, and has no missing values. Most timestamp steps are 10 seconds, with some 9- and 12-second steps and a few long gaps. The 5-minute averages make the longer-term patterns easier to inspect.

- **Common operating modes:** The most frequent state combination is `COMP=1`, `DV_eletric=0`, and `MPG=1` (about 83% of readings). A second common mode is `COMP=0`, `DV_eletric=1`, and `MPG=0` (about 16%). These line up with offloaded and loaded compressor operation in the signal descriptions.
- **Motor and pressure:** In the loaded mode, motor current averages about 5.7 A and `TP2` is typically raised. In the offloaded mode, motor current is lower on average and `TP2` is close to zero. `TP3` and `Reservoirs` usually move closely together.
- **Slower signal:** Oil temperature changes more slowly than the pressure and state signals, and varies across the months.
- **Different behavior over time:** The proportion of readings in the loaded mode changes by month. June has a higher loaded share than February, while much of the recording continues to alternate between the two common states.
- **Periods to inspect later:** The sustained loaded-state intervals on 29 March, 18 April, 29–30 May, 5–7 June, and the afternoon of 15 July are visible candidates for closer review. In particular, 15 July includes elevated oil temperature and increased `LPS` activity. This is a descriptive shortlist only; it does not establish cause or label these periods.
- **No filtering:** These observations do not remove any rows or determine which data belongs in training. We can make that decision later after defining the training approach and reviewing the operating context.

## 7. Initial Insights

- **Variation:** `TP2`, `H1`, and `Motor_current` have large ranges across the recording. Oil temperature varies more gradually.
- **Correlations:** `TP3` and `Reservoirs` track closely. Use the Pearson table above for other strong or weak pairs. Correlation does not mean causation.
- **Near-constant signals:** No continuous signal is exactly constant; several of the state signals are binary by design.
- **Missing data:** The downloaded CSV has no missing values.
- **Operating behavior:** Most readings fall into the offloaded or loaded state combinations described above, with pressure and motor current changing alongside state.
- **Periods for later review:** See the dates in Operating Behavior Observations. They are not labeled as anomalies or failures.
- **Sampling:** The median interval is 10 seconds, though intervals vary and a few large gaps occur.